## **Dependencies**

In [1]:
import os
import sys
import json
import glob
import joblib

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from transformers import AutoTokenizer, AutoModelForCausalLM

import xgboost as xgb
import shap
from PIL import Image

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

from huggingface_hub import login
login(token=hf_token)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## **Configurations**

In [22]:
# Raw datasets (original data, for pulling sample inputs)
MALMEM_CSV        = "/kaggle/input/datasets/bertvankeulen/cic-malmem-2022/MalMem2022.csv"
API_CSV           = "/kaggle/input/datasets/ang3loliveira/malware-analysis-datasets-api-call-sequences/dynamic_api_call_sequence_per_malware_100_0_306.csv"
MALEVIS_DIR       = "/kaggle/input/datasets/sohamkumar1703/malevis-dataset/malevis_train_val_300x300"

# Artifact datasets (model files + preprocessing pipelines)
ARTIFACTS_XGB      = "/kaggle/input/datasets/anarvaaa/artifacts-for-xgboost"
ARTIFACTS_TRANSFORMER = "/kaggle/input/datasets/anarvaaa/artifacts-for-transformer"
ARTIFACTS_CNN = "/kaggle/input/datasets/anarvaaa/artifacts-for-cnn"

# Security Documents
MITRE_ATTACK_JSON = r"/kaggle/input/datasets/anarvaaa/security-documentations/enterprise-attack-19.1.json"
LOCAL_DOCS = "/kaggle/input/datasets/anarvaaa/security-documentations/security_docs.json"

# LLM Model and HF tokens
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
HF_TOKEN = hf_token

In [5]:
# XGBoost Artifact Paths
xgb_pipeline_path = os.path.join(ARTIFACTS_XGB, "preprocessing_pipeline.joblib")
xgb_encoder_path  = os.path.join(ARTIFACTS_XGB, "label_encoder.joblib")
xgb_model_path    = os.path.join(ARTIFACTS_XGB, "calibrated_xgboost_model.joblib")

# Transformer Artifact Paths
transformer_pipeline_path     = os.path.join(ARTIFACTS_TRANSFORMER, "transformer_preprocessing_pipeline.json")
transformer_model_path        = os.path.join(ARTIFACTS_TRANSFORMER, "api_transformer_model.pth")
transformer_platt_scalar_path = os.path.join(ARTIFACTS_TRANSFORMER, "transformer_platt_scaler.joblib")

# CNN Artifacts Paths
cnn_pipeline_path    = os.path.join(ARTIFACTS_CNN, "cnn_preprocessing_pipeline.json")
cnn_model_path       = os.path.join(ARTIFACTS_CNN, "malevis_cnn_model.pth")
cnn_calibration_path = os.path.join(ARTIFACTS_CNN, "cnn_platt_calibrators.joblib")
cnn_class_weights    = os.path.join(ARTIFACTS_CNN, "cnn_class_weights.json")

In [23]:
# Helper Functions
sys.path.append("/kaggle/input/datasets/anarvaaa/utilities")

from inference_functions import (
    xgboost_calibrated_inference,
    transformer_calibrated_inference,
    cnn_calibrated_inference,
    device,
)

from combining_logic import total_maliciousness

from fixed_config import (
    max_limit_files, per_class_limit, hierarchy,
    xgb_per_class_fscores, xgb_weighted_f1score,
    transformer_per_class_fscores, transformer_weighted_f1score,
    API_Calls,
)

from security_docs import load_security_references, assemble_context_snippets

from prompt_generation import generate_soc_report

from json_summary_generator import build_final_summary

## **Initializations**

In [7]:
# Converting Malevis directory to list-like format to grab a file by index later (CNN)
sample_image_candidates = glob.glob(os.path.join(MALEVIS_DIR, "**", "*.png"), recursive=True)
if not sample_image_candidates:
    sample_image_candidates = glob.glob(os.path.join(MALEVIS_DIR, "**", "*.jpg"), recursive=True)

# Loading Malmem 2022 dataset (xgboost)
df_malmem = pd.read_csv(MALMEM_CSV)

# Loading API Call dataset (transformer)
df_api = pd.read_csv(API_CSV)
seq_cols = [c for c in df_api.columns if c.startswith('t_')]

In [8]:
# Load the precomputed w_normalized class weights (saved by the training notebook),
# so score_calc can look them up by class name -- cnn_class_weights above is just the
# file PATH, this loads its actual contents.
with open(cnn_class_weights, "r") as f:
    cnn_class_weights_data = json.load(f)

In [19]:
# Load MITRE ATT&CK + local SECURITY_DOCS grounding references
# (indexes are built inside security_docs.py; LOCAL_DOCS points at the
# security_docs.json file uploaded in the security_documentations dataset)
load_security_references(MITRE_ATTACK_JSON, LOCAL_DOCS)

Loaded MITRE ATT&CK data: 25843 STIX objects
MITRE indexes ready: 1581 techniques, 992 software entries, 602 group entries
Loaded local SECURITY_DOCS: 24 entries across 3 categories


In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto",
)

model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size

print(
    f"Loaded {MODEL_NAME}: "
    f"{NUM_LAYERS} layers, "
    f"hidden_dim={HIDDEN_DIM}, "
    f"device={DEVICE}"
)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct: 28 layers, hidden_dim=3072, device=cuda:0


## **Input**

In [12]:
#index for: malmem dataset (xgboost), api dataset (transformer), list of indexes from malevis directory
input_indices = [50000,115,[617, 614, 1814, 1815, 1816, 616, 182, 183, 0, 1 ,2, 606, 607]]

In [13]:
xgb_input = df_malmem.iloc[[input_indices[0]]]
transformer_input = df_api[seq_cols].iloc[[input_indices[1]]]
cnn_input = input_indices[2]

## **Predictions**

In [14]:
xgb_results = xgboost_calibrated_inference(
    input_data=xgb_input,
    calibrated_model_path=xgb_model_path,
    pipeline_path=xgb_pipeline_path,
    encoder_path=xgb_encoder_path
)

transformer_results = transformer_calibrated_inference(
    input_sequences=transformer_input,
    model_path=transformer_model_path,
    platt_scaler_path=transformer_platt_scalar_path,
    config_path=transformer_pipeline_path,
    api_mapping = API_Calls
)

cnn_all_results = []

for i in cnn_input:
    cnn_results = cnn_calibrated_inference(
        input_data=sample_image_candidates[i],
        model_path=cnn_model_path,
        pipeline_path=cnn_pipeline_path,
        calibration_path=cnn_calibration_path,
        file_index = i
    )
    cnn_all_results.append(cnn_results)
    

print("XGB RESULTS: \n")
display(xgb_results)
print("\nTRANSFORMER RESULTS: \n")
display(transformer_results)
print("\nCNN RESULTS: \n")
display(cnn_all_results)

XGB RESULTS: 



{'Predicted_Class_ID': array([3]),
 'Predicted_Class': array(['Trojan'], dtype=object),
 'Calibrated_Confidence': array([0.82236778]),
 'Probability_Distribution': array([3.08054034e-04, 1.53588029e-01, 2.37361419e-02, 8.22367775e-01]),
 'Top_Factors': [{'feature': 'handles.nsemaphore',
   'shap_value': 0.729843020439148},
  {'feature': 'handles.nfile', 'shap_value': 0.6047630310058594},
  {'feature': 'handles.nmutant', 'shap_value': 0.17086315155029297}]}


TRANSFORMER RESULTS: 



{'Predicted_Class_ID': array([1]),
 'Predicted_Class': ['Malware'],
 'Calibrated_Confidence': array([0.97952066]),
 'Probability_Distribution': array([0.02047934, 0.97952066]),
 'Top_Factors': [{'timestep': 2,
   'api_call_id': 187,
   'api_call': 'NtFreeVirtualMemory',
   'importance': 0.0385746955871582},
  {'timestep': 46,
   'api_call_id': 172,
   'api_call': 'LdrGetDllHandle',
   'importance': 0.008847832679748535},
  {'timestep': 47,
   'api_call_id': 117,
   'api_call': 'LdrGetProcedureAddress',
   'importance': 0.00988006591796875},
  {'timestep': 48,
   'api_call_id': 172,
   'api_call': 'LdrGetDllHandle',
   'importance': 0.00864708423614502},
  {'timestep': 49,
   'api_call_id': 117,
   'api_call': 'LdrGetProcedureAddress',
   'importance': 0.009826719760894775},
  {'timestep': 50,
   'api_call_id': 172,
   'api_call': 'LdrGetDllHandle',
   'importance': 0.00848454236984253},
  {'timestep': 51,
   'api_call_id': 117,
   'api_call': 'LdrGetProcedureAddress',
   'importance': 


CNN RESULTS: 



[{'Predicted_Class_ID': array([22]),
  'Predicted_Class': ['Stantinko'],
  'Calibrated_Confidence': array([0.87779326]),
  'Probability_Distribution': array([[2.69856757e-09, 2.23661886e-03, 7.46067114e-05, 2.97434847e-03,
          3.78193814e-04, 7.11501811e-04, 2.87407012e-07, 8.23642048e-11,
          9.47419313e-04, 1.38896487e-04, 1.94643559e-09, 7.37674537e-13,
          1.49155582e-06, 6.83128769e-04, 1.73189906e-05, 2.53847937e-03,
          2.43427839e-08, 4.04809294e-05, 1.08510438e-01, 3.24635812e-06,
          2.30252904e-03, 6.89531715e-18, 8.77793259e-01, 2.54432750e-10,
          6.46849872e-04, 8.75129259e-07]]),
  'File_index': 617},
 {'Predicted_Class_ID': array([4]),
  'Predicted_Class': ['Androm'],
  'Calibrated_Confidence': array([0.55798292]),
  'Probability_Distribution': array([[9.84414018e-08, 2.05195244e-03, 1.36034835e-05, 1.00769416e-04,
          5.57982917e-01, 1.77884778e-04, 4.18350355e-02, 6.93667237e-10,
          2.95657724e-04, 1.66093358e-03, 1.338

## **JSON Summary**

In [15]:
base_P_mal,cnn_effect,final_P_mal,disagreement, scored_files = total_maliciousness(
                        xgb_results, transformer_results,
                        cnn_all_results, cnn_class_weights_data,
                        xgb_per_class_fscores, transformer_per_class_fscores,
                        hierarchy, max_limit_files, per_class_limit)

In [24]:
final_summary = build_final_summary(
    base_P_mal, disagreement, final_P_mal,
    xgb_results, transformer_results, cnn_all_results, scored_files
)

display(final_summary)

{'Base_P_malicious': np.float64(0.9981),
 'Disagreement': np.float64(0.0619),
 'CNN_evidence_rise_pct': np.float64(0.1495),
 'Final_P_malicious': np.float64(0.9995),
 'Xgboost_Summary': {'predicted_state': 'Trojan',
  'confidence': np.float64(0.822),
  'supporting_evidence': ['handles.nsemaphore',
   'handles.nfile',
   'handles.nmutant']},
 'Transformer_Summary': {'predicted_state': 'Malware',
  'confidence': np.float64(0.98),
  'supporting_evidence': ['NtFreeVirtualMemory',
   'LdrGetDllHandle',
   'LdrGetProcedureAddress',
   'LdrGetDllHandle',
   'LdrGetProcedureAddress',
   'LdrGetDllHandle',
   'LdrGetProcedureAddress',
   'LdrGetDllHandle',
   'LdrGetProcedureAddress',
   'LoadResource']},
 'CNN_Summary': {'total_scanned_files': 13,
  'influencial_files': [{'file_index': 183,
    'predicted_class': 'HackKMS',
    'confidence': 0.973},
   {'file_index': 182, 'predicted_class': 'HackKMS', 'confidence': 0.971},
   {'file_index': 617, 'predicted_class': 'Stantinko', 'confidence': 0.

## **SOC Analyst Report**

In [20]:
# --- Run ---
soc_report, matched_snippets, unmatched_labels = generate_soc_report(final_summary, tokenizer, model, DEVICE)

print("REFERENCE DOCS MATCHED:\n")
for s in matched_snippets:
    print(s, "\n")

if unmatched_labels:
    print("NO REFERENCE AVAILABLE FOR:", unmatched_labels, "\n")

print("="*80 + "\nSOC ANALYST REPORT\n" + "="*80 + "\n")
print(soc_report)

REFERENCE DOCS MATCHED:

[RAM state: Trojan]
  Summary: Memory artifacts consistent with a trojan.
  Indicators: Malware disguised as legitimate software; commonly used to open a backdoor, disable defenses, or drop further payloads.
  Suggested response: Isolate the host, capture a full memory image, and inventory any newly dropped executables or scheduled tasks. 

[File family: HackKMS]
  Summary: Bundled activation-cracker tool.
  Indicators: Commonly bundled with pirated Windows/Office activation cracks; frequently repackaged with additional malicious payloads.
  Suggested response: Identify how the file arrived (download source) and check for other software installed around the same time. 

[File family: Stantinko]
  Summary: Adware/botnet family.
  Indicators: Persistence via hidden Windows services; ad-fraud and crypto-mining side effects.
  Suggested response: Check for unexpected background services and unusual outbound ad-network traffic. 

[File family: Amonetize]
  Summary: 